In [1]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import tushare as ts
import json
import time
from datetime import datetime, timedelta
from dotenv import load_dotenv

# 加载环境变量
env_path = os.path.join( '../.env')
load_dotenv(env_path)
ts.set_token(os.environ.get('TUSHARE_TOKEN'))
pro = ts.pro_api()

DB_W = os.path.join('warehouse.db')

In [6]:
conn = sqlite3.connect(DB_W)

    # 1. 寻找数据最完整的最近一个交易日作为基准 (防止被半路同步的今天数据干扰)
latest_date_row = conn.execute("""
    SELECT trade_date FROM historical_daily 
    WHERE pe_ttm IS NOT NULL 
    GROUP BY trade_date 
    HAVING COUNT(*) > 1000 
    ORDER BY trade_date DESC 
    LIMIT 1
""").fetchone()
latest_date = latest_date_row[0] if latest_date_row else None

if not latest_date:
    print("Error: No complete trading day found in warehouse (>1000 rows). Please run catch-up sync first.")
    conn.close()
    
print(f"Using baseline date: {latest_date}")
# 1. VIT, STR, AGI (从该完整日计算分位)
df_basic = pd.read_sql_query(f"SELECT ts_code, pe_ttm, dv_ttm, turnover_rate_f FROM historical_daily WHERE trade_date = '{latest_date}'", conn)


Using baseline date: 20260414


In [7]:
df_basic.describe()

,pe_ttm,dv_ttm,turnover_rate_f
count,4010.000000,3750.000000,5496.000000
mean,125.823291,1.442488,5.286579
std,415.737081,1.496649,5.724115
min,3.963500,0.009300,0.173000
25%,25.164825,0.403525,2.032800
50%,46.747250,0.911950,3.315800
75%,101.486225,1.945100,6.141150
max,10663.997400,13.504200,62.517900


In [10]:
from update_rpg_benchmarks import get_percentile_table

In [11]:
get_percentile_table( 100.0 / df_basic['pe_ttm'])

[0.00954676330689685,
 0.07645335084161638,
 0.13597490520444674,
 0.17443628058046742,
 0.20364564351585376,
 0.25693871516153627,
 0.2914426381768241,
 0.33188796515444163,
 0.36858702357310164,
 0.4064029519017105,
 0.4448948612231701,
 0.4831324416409623,
 0.5229846418292359,
 0.5681097161058836,
 0.590061757505402,
 0.6232050006072468,
 0.6538140493325687,
 0.6978948062967426,
 0.7377614786869301,
 0.7704884173711204,
 0.8241778358083233,
 0.8598246174732093,
 0.890554319188258,
 0.9275286972490053,
 0.9566166013159051,
 0.9852672980916357,
 1.0302681187264733,
 1.077760320135477,
 1.119798883251493,
 1.1537782471519105,
 1.187307928416362,
 1.2276382187969797,
 1.2710788352445628,
 1.3248065996809584,
 1.3723501479489892,
 1.4167082820647288,
 1.4679226285699791,
 1.5126008417763037,
 1.5527266694983008,
 1.600907204811607,
 1.65809602953482,
 1.6990042681881534,
 1.7571584896591237,
 1.8121005847722222,
 1.8673339850817792,
 1.9192237007476847,
 1.9748781378349083,
 2.0148971294